<a href="https://colab.research.google.com/github/anasmita3/SURGE/blob/main/SURGE_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install datasets

Literature

In [ ]:
from huggingface_hub import login

login("")

In [ ]:
from datasets import load_dataset

ds = load_dataset("Vacaspati/Vacaspati")

print(ds)

print("\nColumns:")
print(ds["train"].column_names)

print("\nSample:")
print(ds["train"][0])

In [ ]:
import unicodedata
import re

count = 0

with open("bn_literature.txt", "w", encoding="utf-8") as out:

    for row in ds["train"]:

        text = row["text"]

        text = unicodedata.normalize("NFC", text)
        text = re.sub(r"\s+", " ", text).strip()

        if not text:
            continue

        out.write(text + "\n")
        count += 1

print("Saved documents:", count)

In [ ]:
import random

TARGET_WORDS = 5601171

with open("bn_literature.txt", encoding="utf-8") as f:
    docs = [x.strip() for x in f if x.strip()]

random.seed(42)
random.shuffle(docs)

selected = []
word_count = 0

for doc in docs:
    wc = len(doc.split())

    if word_count + wc > TARGET_WORDS:
        continue

    selected.append(doc)
    word_count += wc

    if word_count >= TARGET_WORDS:
        break

print("Selected documents:", len(selected))
print("Selected words:", word_count)

with open(
    "literature_balanced.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("\n".join(selected))

In [ ]:
from collections import Counter

file_path = "literature_balanced.txt"

docs = 0
words = 0

with open(file_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            docs += 1
            words += len(line.split())

print("Documents:", docs)
print("Words:", words)

In [ ]:
from collections import Counter

file_path = "literature_balanced.txt"

docs = 0
words = 0
chars = 0
vocab = Counter()

with open(file_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        docs += 1
        chars += len(line)

        ws = line.split()
        words += len(ws)

        vocab.update(ws)

print("Documents :", docs)
print("Characters:", chars)
print("Words     :", words)
print("Vocabulary:", len(vocab))
print("Avg Words/Doc:", round(words/docs, 2))

News

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "ai4bharat/IndicCorpV2",
    repo_type="dataset"
)

print(len(files))
print(files[:20])

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    data_files={"train": "data/bn.txt"}
)

print(dataset)

In [ ]:
import unicodedata
import re

MAX_DOCS = 500000

with open("bn_news.txt", "w", encoding="utf-8") as f:

    for i, row in enumerate(dataset["train"]):

        text = unicodedata.normalize("NFC", row["text"])
        text = re.sub(r"\s+", " ", text).strip()

        if text:
            f.write(text + "\n")

        if i + 1 >= MAX_DOCS:
            break

print("Saved", MAX_DOCS, "documents")

In [ ]:
import random

random.seed(42)

TARGET = 125000

with open("bn_news.txt", encoding="utf-8") as f:
    total_docs = sum(1 for _ in f)

print("Total news docs:", total_docs)

selected = set(
    random.sample(
        range(total_docs),
        TARGET
    )
)

count = 0

with open("bn_news.txt", encoding="utf-8") as inp, \
     open("news_125k.txt", "w", encoding="utf-8") as out:

    for idx, line in enumerate(inp):

        if idx in selected:
            out.write(line)
            count += 1

print("Saved:", count)

In [ ]:
from collections import Counter

file_path = "news_125k.txt"

docs = 0
words = 0
chars = 0
vocab = Counter()

with open(file_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        docs += 1
        chars += len(line)

        ws = line.split()
        words += len(ws)

        vocab.update(ws)

print("Documents :", docs)
print("Characters:", chars)
print("Words     :", words)
print("Vocabulary:", len(vocab))
print("Avg Words/Doc:", round(words/docs, 2))

Merge

In [ ]:
import random

# Load news
with open("news_125k.txt", encoding="utf-8") as f:
    news = [x.strip() for x in f if x.strip()]

# Load literature
with open("literature_balanced.txt", encoding="utf-8") as f:
    literature = [x.strip() for x in f if x.strip()]

print("News docs      :", len(news))
print("Literature docs:", len(literature))

# Merge
merged = news + literature

# Shuffle
random.seed(42)
random.shuffle(merged)

# Save
with open(
    "news_literature_merged.txt",
    "w",
    encoding="utf-8"
) as f:
    for doc in merged:
        f.write(doc + "\n")

print("Merged docs:", len(merged))

In [ ]:
from collections import Counter

chars = 0
words = 0
docs = 0
vocab = Counter()

with open(
    "news_literature_merged.txt",
    encoding="utf-8"
) as f:

    for line in f:
        line = line.strip()

        if not line:
            continue

        docs += 1
        chars += len(line)

        ws = line.split()

        words += len(ws)
        vocab.update(ws)

print("Documents :", docs)
print("Characters:", chars)
print("Words     :", words)
print("Vocabulary:", len(vocab))
print("TTR       :", len(vocab)/words)
print("Average Word Length:", chars/words)

The merged News–Literature corpus contains 376,038 documents and approximately 11.26 million words. Although the number of documents differs between the two domains, their word counts are nearly identical, ensuring balanced lexical contributions during tokenizer training. The corpus exhibits a large vocabulary of 827,293 unique words, reflecting the complementary linguistic characteristics of formal news text and literary writing. The reduced Type–Token Ratio (0.0735) indicates increased lexical stability due to the larger corpus size, while the average word length remains consistent with observations from the individual corpora.


Transliteration

In [ ]:
!pip install aksharamukha
from aksharamukha import transliterate
from tqdm import tqdm

INPUT_FILE = "news_literature_merged.txt"
OUTPUT_FILE = "news_literature_iso.txt"

with open(INPUT_FILE, encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for line in tqdm(fin):
        line = line.strip()

        if not line:
            continue

        iso = transliterate.process(
            "Bengali",
            "ISO",
            line
        )

        fout.write(iso + "\n")

print("Done.")

In [ ]:
from collections import Counter

def corpus_stats(path):

    chars = 0
    words = 0
    vocab = Counter()

    with open(path, encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            chars += len(line)

            ws = line.split()

            words += len(ws)
            vocab.update(ws)

    return {
        "chars": chars,
        "words": words,
        "vocab": len(vocab),
        "avg_word_len": chars / words
    }

bn = corpus_stats("news_literature_merged.txt")
iso = corpus_stats("news_literature_iso.txt")

print("Bengali:", bn)
print("ISO:", iso)

print()
print("Expansion Factor:",
      iso["chars"] / bn["chars"])

print("Vocabulary Ratio:",
      iso["vocab"] / bn["vocab"])

In [ ]:
from aksharamukha import transliterate
import random

with open(
    "news_literature_merged.txt",
    encoding="utf-8"
) as f:

    docs = [
        x.strip()
        for x in f
        if x.strip()
    ]

random.seed(42)

sample = random.sample(
    docs,
    100
)

correct = 0

for txt in sample:

    iso = transliterate.process(
        "Bengali",
        "ISO",
        txt
    )

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso
    )

    if txt == back:
        correct += 1

print(
    f"Exact Match: {correct} / 100"
)

print(
    "Accuracy:",
    correct / 100
)

In [ ]:
from sklearn.model_selection import train_test_split

with open(
    "news_literature_merged.txt",
    encoding="utf-8"
) as f:

    docs = [
        x.strip()
        for x in f
        if x.strip()
    ]

train_docs, test_docs = train_test_split(
    docs,
    test_size=0.10,
    random_state=42
)

with open(
    "train_nl_bn.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("\n".join(train_docs))

with open(
    "test_nl_bn.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("\n".join(test_docs))

print("Train:", len(train_docs))
print("Test :", len(test_docs))

In [ ]:
from sklearn.model_selection import train_test_split

with open(
    "news_literature_iso.txt",
    encoding="utf-8"
) as f:

    docs = [
        x.strip()
        for x in f
        if x.strip()
    ]

train_docs, test_docs = train_test_split(
    docs,
    test_size=0.10,
    random_state=42
)

with open(
    "train_nl_iso.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("\n".join(train_docs))

with open(
    "test_nl_iso.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("\n".join(test_docs))

print("Train:", len(train_docs))
print("Test :", len(test_docs))

In [ ]:
import os

for f in [
    "train_nl_bn.txt",
    "test_nl_bn.txt",
    "train_nl_iso.txt",
    "test_nl_iso.txt"
]:

    print(
        f,
        round(
            os.path.getsize(f)/(1024*1024),
            2
        ),
        "MB"
    )

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tok.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_nl_bn.txt"],
        trainer
    )

    tok.save(
        f"bpe_bn_{vocab_size}.json"
    )

    print(vocab_size)

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tok.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_nl_iso.txt"],
        trainer
    )

    tok.save(
        f"bpe_iso_{vocab_size}.json"
    )

    print(vocab_size)

In [ ]:
from tokenizers import Tokenizer
import numpy as np

def evaluate_tokenizer(tokenizer_path, test_file):

    tok = Tokenizer.from_file(tokenizer_path)

    total_words = 0
    total_tokens = 0

    oov = 0

    token_counts = []

    with open(test_file, encoding="utf-8") as f:

        for line in f:

            ws = line.strip().split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                total_words += 1
                total_tokens += len(tks)

                token_counts.append(
                    len(tks)
                )

                if "[UNK]" in tks:
                    oov += 1

    fertility = total_tokens / total_words

    return {
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": 1 / fertility,
        "compression": 1 / fertility,
        "avg_tokens": fertility,
        "wfr": fertility - 1,
        "variance": np.var(token_counts)
    }

In [ ]:
VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_bn_{vocab_size}.json",
        "test_nl_bn.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

In [ ]:
for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_iso_{vocab_size}.json",
        "test_nl_iso.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

### BPE Evaluation

BPE exhibited consistent improvements as vocabulary size increased from 5k to 50k. Fertility decreased steadily, indicating that larger vocabularies produced longer and more informative subword units. OOV rates remained effectively zero across all configurations, demonstrating strong vocabulary coverage.

For both Bengali and ISO representations, tokenization quality improved substantially between 5k and 30k vocabulary sizes, after which gains became progressively smaller. This suggests diminishing returns beyond approximately 30k–40k vocabulary.

Interestingly, the Bengali representation achieved slightly lower fertility and fragmentation than the ISO representation at larger vocabulary sizes, although the differences were marginal. The results indicate that BPE effectively captures lexical and morphological patterns present in both news and literary text while maintaining stable segmentation behavior.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tok.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_nl_bn.txt"],
        trainer
    )

    tok.save(
        f"wp_bn_{vocab_size}.json"
    )

    print(vocab_size)

print("Finished.")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tok.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tok.train(
        ["train_nl_iso.txt"],
        trainer
    )

    tok.save(
        f"wp_iso_{vocab_size}.json"
    )

    print(vocab_size)

print("Finished.")

In [ ]:
VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_bn_{vocab_size}.json",
        "test_nl_bn.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

In [ ]:
VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_iso_{vocab_size}.json",
        "test_nl_iso.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

In [ ]:
import pandas as pd

bn_results = []
iso_results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_bn_{vocab_size}.json",
        "test_nl_bn.txt"
    )
    r["vocab"] = vocab_size
    bn_results.append(r)

    r = evaluate_tokenizer(
        f"wp_iso_{vocab_size}.json",
        "test_nl_iso.txt"
    )
    r["vocab"] = vocab_size
    iso_results.append(r)

pd.DataFrame(bn_results).to_csv(
    "wp_bn_results.csv",
    index=False
)

pd.DataFrame(iso_results).to_csv(
    "wp_iso_results.csv",
    index=False
)

print("Saved.")

### WordPiece Evaluation

WordPiece demonstrated the expected reduction in fertility and fragmentation as vocabulary size increased. OOV rates remained negligible across all experiments, indicating strong vocabulary coverage.

Compared with BPE, WordPiece consistently produced higher fertility and word fragmentation rates, suggesting a greater tendency to split words into smaller subword units. Although performance improved substantially between 5k and 30k vocabularies, gains became progressively smaller beyond 30k.

Across both Bengali and ISO representations, WordPiece exhibited slightly higher variance than BPE, indicating less stable segmentation behavior. Overall, WordPiece achieved competitive performance but remained consistently less efficient than BPE on the News–Literature corpus.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(Unigram())

    tok.pre_tokenizer = Whitespace()

    trainer = UnigramTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"],
        unk_token="[UNK]"
    )

    tok.train(
        ["train_nl_bn.txt"],
        trainer
    )

    tok.save(
        f"uni_bn_{vocab_size}.json"
    )

    print(vocab_size)

print("Finished.")

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tok = Tokenizer(Unigram())

    tok.pre_tokenizer = Whitespace()

    trainer = UnigramTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"],
        unk_token="[UNK]"
    )

    tok.train(
        ["train_nl_iso.txt"],
        trainer
    )

    tok.save(
        f"uni_iso_{vocab_size}.json"
    )

    print(vocab_size)

print("Finished.")

In [ ]:
from tokenizers import Tokenizer
import numpy as np

def evaluate_tokenizer(tokenizer_path, test_file):

    tok = Tokenizer.from_file(tokenizer_path)

    total_words = 0
    total_tokens = 0

    oov = 0

    token_counts = []

    with open(test_file, encoding="utf-8") as f:

        for line in f:

            ws = line.strip().split()

            for w in ws:

                try:
                    enc = tok.encode(w)
                    tks = enc.tokens

                except:
                    oov += 1
                    total_words += 1
                    token_counts.append(1)
                    continue

                total_words += 1
                total_tokens += len(tks)

                token_counts.append(
                    len(tks)
                )

                if "[UNK]" in tks:
                    oov += 1

    fertility = total_tokens / total_words

    return {
        "oov": oov / total_words,
        "fertility": fertility,
        "cpt": 1 / fertility,
        "compression": 1 / fertility,
        "avg_tokens": fertility,
        "wfr": fertility - 1,
        "variance": np.var(token_counts)
    }

In [ ]:
VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"uni_bn_{vocab_size}.json",
        "test_nl_bn.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

In [ ]:
VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"uni_iso_{vocab_size}.json",
        "test_nl_iso.txt"
    )

    r["vocab"] = vocab_size

    print(r)

print("\nFinished.")

In [ ]:
import random

random.seed(42)

with open(
    "test_nl_bn.txt",
    encoding="utf-8"
) as f:

    sentences = [
        x.strip()
        for x in f
        if x.strip()
    ]

sample_sentences = random.sample(
    sentences,
    20
)

for i,s in enumerate(sample_sentences,1):
    print(f"{i}. {s}")

In [ ]:
from tokenizers import Tokenizer

bpe = Tokenizer.from_file(
    "bpe_bn_50000.json"
)

wp = Tokenizer.from_file(
    "wp_bn_50000.json"
)

uni = Tokenizer.from_file(
    "uni_bn_50000.json"
)

In [ ]:
for i,sent in enumerate(sample_sentences,1):

    print("="*80)
    print(f"Sentence {i}")
    print(sent)

    print("\nBPE")
    print(bpe.encode(sent).tokens)

    print("\nWordPiece")
    print(wp.encode(sent).tokens)

    print("\nUnigram")
    print(uni.encode(sent).tokens)

In [ ]:
import pandas as pd

rows = []

for sent in sample_sentences:

    rows.append({
        "Sentence": sent,
        "BPE": " ".join(
            bpe.encode(sent).tokens
        ),
        "WordPiece": " ".join(
            wp.encode(sent).tokens
        ),
        "Unigram": " ".join(
            uni.encode(sent).tokens
        )
    })

df = pd.DataFrame(rows)

df.to_csv(
    "morphological_analysis.csv",
    index=False
)

df.head()

In [ ]:
def fertility_on_sentences(tok, sentences):

    words = 0
    pieces = 0

    for s in sentences:

        for w in s.split():

            pieces += len(
                tok.encode(w).tokens
            )

            words += 1

    return pieces / words

print(
    "BPE:",
    fertility_on_sentences(
        bpe,
        sample_sentences
    )
)

print(
    "WordPiece:",
    fertility_on_sentences(
        wp,
        sample_sentences
    )
)

print(
    "Unigram:",
    fertility_on_sentences(
        uni,
        sample_sentences
    )
)

In [ ]:
import os

for f in os.listdir():
    print(f)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import nbformat

nb = nbformat.read(
    "SURGE_9.ipynb",
    as_version=4
)

nbformat.write(
    nb,
    "SURGE_9_fixed.ipynb"
)

print("done")

In [ ]:
from google.colab import files
files.download("SURGE_9_fixed.ipynb")

In [ ]:
import nbformat

INPUT = "SURGE_9.ipynb"
OUTPUT = "SURGE_9_fixed.ipynb"

with open(INPUT, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

# Remove problematic widget metadata only
if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

with open(OUTPUT, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Fixed.")